In [1]:
import sys
import pandas as pd
import numpy as np
import pyarrow

from sklearn.model_selection import train_test_split

print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("PyArrow:", pyarrow.__version__)

Python: c:\Users\kinga\anaconda3\envs\machinelearning\python.exe
Pandas: 3.0.6
NumPy: 2.5.4
PyArrow: 26.0.0


In [2]:
df = pd.read_parquet("../data/train-00000-of-00001.parquet")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (124146, 41)


,appID,name,release_date,estimated_owners,peak_ccu,required_age,price,dlc_count,detailed_description,short_description,...,median_playtime_forever,median_playtime_2weeks,developers,publishers,categories,genres,tags,screenshots,movies,packages
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,,,...,0,0,[],[],[],[],[],[https://shared.akamai.steamstatic.com/store_i...,[],[]
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,0,"Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",...,8,0,[minori],[MangaGamer],"[Single-player, Steam Trading Cards, Steam Clo...",[Adventure],[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Supipara - Chapter 1 Spring Ha..."
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,"Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,...,0,0,[Somer Games],[8floor],"[Single-player, Family Sharing]",[Casual],[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Mystery Solitaire The Black Ra..."
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",Yuha! I'll start the broadcast! Hakko's extrem...,...,0,0,[유진게임즈],[유진게임즈],"[Single-player, Steam Achievements, Family Sha...","[Casual, Indie, Simulation]",[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy \ubc84\ud29c\ubc84 \ud30c\ub77..."
4,3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,...,0,0,[Reality Expanded LLC],[Reality Expanded LLC],"[Single-player, VR Only, Steam Leaderboards, F...","[Action, Early Access]",[],[https://shared.akamai.steamstatic.com/store_i...,[],"[{""title"": ""Buy Maze Quest VR"", ""description"":..."


In [3]:
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

print("\nRelease dates:")
print(df["release_date"].head())

<class 'pandas.DataFrame'>
RangeIndex: 124146 entries, 0 to 124145
Data columns (total 41 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   appID                     124146 non-null  str    
 1   name                      124146 non-null  str    
 2   release_date              124146 non-null  str    
 3   estimated_owners          124146 non-null  str    
 4   peak_ccu                  124146 non-null  int64  
 5   required_age              124146 non-null  int64  
 6   price                     124146 non-null  float64
 7   dlc_count                 124146 non-null  int64  
 8   detailed_description      124146 non-null  str    
 9   short_description         124146 non-null  str    
 10  supported_languages       124146 non-null  object 
 11  full_audio_languages      124146 non-null  object 
 12  reviews                   124146 non-null  str    
 13  header_image              124146 non-null  str    
 14 

In [4]:
df["release_date"] = pd.to_datetime(
    df["release_date"],
    errors="coerce"
)

df["release_year"] = df["release_date"].dt.year

In [5]:
df = df[
    (df["release_year"] >= 2006) &
    (df["release_year"] <= 2024)
].copy()

df = df.dropna(subset=["positive", "release_year"])

print("Games after filtering:", len(df))

Games after filtering: 97528


In [6]:
df["rank_in_year"] = (
    df.groupby("release_year")["positive"]
    .rank(method="first", ascending=False)
)

df["games_in_year"] = (
    df.groupby("release_year")["positive"]
    .transform("count")
)

df["hit"] = (
    df["rank_in_year"] <= np.ceil(df["games_in_year"] * 0.25)
).astype(int)

print(df[["name", "release_year", "positive", "hit"]].head(10))

print("\nHit distribution:")
print(df["hit"].value_counts())

print("\nHit percentage:")
print(df["hit"].value_counts(normalize=True) * 100)

                                     name  release_year  positive  hit
0              Black Dragon Mage Playtest          2023         0    0
1   Supipara - Chapter 1 Spring Has Come!          2016       252    1
2       Mystery Solitaire The Black Raven          2019        21    0
3             버튜버 파라노이아 - Vtuber Paranoia          2024         0    0
5                                Agony VR          2023         0    0
7          Galacatraz: Eject Equip Escape          2018         0    0
8                             Hepta Beats          2021        28    0
9                  MUMBA IV: Egypt Jewels          2021        19    0
10             Codename: Warlock Playtest          2022         0    0
11                                OMNIMUS          2019        47    0

Hit distribution:
hit
0    73138
1    24390
Name: count, dtype: int64

Hit percentage:
hit
0    74.991797
1    25.008203
Name: proportion, dtype: float64


One limitation of the target creation method is how tied positive review counts are handled. The ranking uses `method="first"`, meaning that games with the same number of positive reviews receive different ranks based on their order in the dataset. This could affect which games are classified as hits near the top-25% cutoff. Therefore, the target labels may be slightly affected by the original row order.

In [7]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.4,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

print("\nPercentages:")

print("Training:", len(train_df) / len(df) * 100)
print("Validation:", len(val_df) / len(df) * 100)
print("Testing:", len(test_df) / len(df) * 100)

Training: 58516
Validation: 19506
Testing: 19506

Percentages:
Training: 59.99917972274629
Validation: 20.000410138626854
Testing: 20.000410138626854


In [8]:
train_df.to_parquet("../data/train.parquet", index=False)
val_df.to_parquet("../data/validation.parquet", index=False)
test_df.to_parquet("../data/test.parquet", index=False)

print("Datasets successfully saved!")

Datasets successfully saved!


In [9]:
train_df = pd.read_parquet("../data/train.parquet")
val_df = pd.read_parquet("../data/validation.parquet")
test_df = pd.read_parquet("../data/test.parquet")

print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Testing:", test_df.shape)

Training: (58516, 45)
Validation: (19506, 45)
Testing: (19506, 45)
